# Procesamiento de Datos en una Infraestructura Cloud 
##Evidencia de Aprendizaje 2

## Integrantes:
### Jorge Armando Rodriguez
### Lina Johana Seguro Gaviria


# 1. DISEÑO DEL ESQUEMA

## a. Descripción Data Set : 
 Ventas de equipamiento de fútbol americano

## Entidades Principales:
![Ventas](./1.png)

![Producto](./2.png)

![Cliente](./3.png)

![Tienda](./4.png)






## b. Esquema SportZone
![Diagrama del esquema](./Esquema.png)

## c. DDL Spark SQL del esquema. 

Se adjunta el archivo .sql que contiene los DDL para construcccion del esquema mostrado. 
[Archivo DDL.sql](./DDL.sql)

# 2. CONFIGURACIÓN DE DATABRICKS

## a. Crear y Configurar un Cluster 

1. En la barra lateral izquierda de tu espacio de trabajo de Databricks, haz clic en el icono de "Compute".
2. Haz clic en "Create Cluster".
3. Configura:
 * Cluster Name: EA3_Actividad2
 * Databricks Runtime Version: e.g., 13.0 LTS (Scala 2.12, Spark 3.4.0)
 * Python Version: e.g., 3.12.3
 * Cluster Mode: Standard
 * Autoscaling: Activado (min 1 nodo – max 4 nodos)
 * Worker Type: e.g., 4 vCPU / 16 GB RAM
4. Haz clic en Create Cluster.

📌 Esto creará un clúster escalable listo para ejecutar Spark y SQL.

📌 Como no es posible realizarlo en la versión Free de Databricks estos serían los resultados. 

## Configuración del Clúster
- Nombre del clúster: EA3_Actividad2
- Databricks Runtime: 13.0 LTS
- Python: 3.12.3
- Modo: Standard
- Núcleos/RAM: 4 vCPU / 16 GB RAM
- Autoscaling: 1–4 nodos


## b. Version de Python y Spark
Configuración del SparkContext

Si ejecutáramos el siguiente código:

```python for item in spark.sparkContext.getConf().getAll(): print(item)```

Esto imprimirá: versión de Spark, configuración de Python, directorios, núcleos y RAM asignados.

Como no es posible en la versión free, solo mostraremos la versión de Spark y Python.

## Versión de Spark
Se imprime usando `spark.version`

## Versión de pytho 
Se imprime usando `sys.version`


In [0]:
# Versión de Spark
print("Versión de Spark:", spark.version)




Versión de Spark: 4.0.0


In [0]:
import sys

# Mostrar versión de Python
print("Versión de Python:", sys.version)




Versión de Python: 3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]


## c. Estructura de almacenamiento
* Volumes (Unity Catalog)
* Se utiliza DBFS en `/FileStore/datasets/` para cargar datos de Kaggle.

##  Obtención de datos de Kaggle 


In [0]:
!pip install kagglehub[pandas-datasets]>=0.3.8

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


##Importaciones de Librerias 

In [0]:
import os
import zipfile
import kagglehub
import pandas as pd 

## Funciones para Descargar, Extraer y el Leer el Dataset desde Kaggle.

In [0]:
def download_dataset_zip(url = ""):
        print("Descargando dataset desde Kaggle...")
        dataset_path = kagglehub.dataset_download(url)
        print("Ruta al dataset:", dataset_path)
        return dataset_path
    
def extract_zip_files(dataset_path):
        zip_files = [f for f in os.listdir(dataset_path) if f.endswith('.zip')]
        if zip_files:
            zip_file = os.path.join(dataset_path, zip_files[0])
            extract_dir = os.path.join(dataset_path, "extracted")
            os.makedirs(extract_dir, exist_ok=True)
            print(f"Extrayendo {zip_file} en {extract_dir}...")
            with zipfile.ZipFile(zip_file, "r") as z:
                z.extractall(extract_dir)
            return extract_dir
        else:
            # Si no se encuentra un ZIP, se verifica si existen archivos CSV en la ruta
            csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]
            if csv_files:
                print("No se encontró archivo ZIP pero se detectaron archivos CSV; se asume que el dataset ya se encuentra extraído.")
                return dataset_path
            else:
                raise FileNotFoundError("No se encontró ningún archivo .zip ni archivos .csv en la ruta del dataset")

def create_csv(csv_dir, csv_name=None):
    if csv_name:
        file_path = os.path.join(csv_dir, csv_name)
        print(f"Leyendo {file_path}...")
        df = pd.read_csv(file_path, encoding="latin1")
        print("CSV creado correctamente")
        return df
    else:
        csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError("No se encontraron archivos CSV en el directorio extraído")
        for file in csv_files:
            file_path = os.path.join(csv_dir, file)
            print(f"Leyendo {file_path}...")
            df = pd.read_csv(file_path, encoding="latin1")
        print("CSV creado correctamente")
        return df

    

## Descarga del Dataset

In [0]:
df = pd.DataFrame()
dataset_path = download_dataset_zip("larysa21/retail-data-american-football-gear-sales") 
csv_dir = extract_zip_files(dataset_path)
df = create_csv(csv_dir, csv_name="AF_offline_sales_dataset.csv")


Descargando dataset desde Kaggle...


100%|██████████| 97.3M/97.3M [00:02<00:00, 44.5MB/s]

Extracting files...


Ruta al dataset: /home/spark-c1ccd4b2-fe89-4364-9290-9b/.cache/kagglehub/datasets/larysa21/retail-data-american-football-gear-sales/versions/1
No se encontró archivo ZIP pero se detectaron archivos CSV; se asume que el dataset ya se encuentra extraído.
Leyendo /home/spark-c1ccd4b2-fe89-4364-9290-9b/.cache/kagglehub/datasets/larysa21/retail-data-american-football-gear-sales/versions/1/AF_offline_sales_dataset.csv...
CSV creado correctamente


In [0]:
df.head(2)

,product_name,brand,category,subcategory,supplier,date,price,quantity_sold,amount_sold,cost_amount,payment_method,customer_firstname,customer_lastname,customer_gender,customer_email,customer_phone,store_type,store_street,store_city,store_state
0,Riddell Victor-I Inflation Air Bladder (R91229...,Riddell,Helmets,Helmet Components,Balistreri Inc.,2023-07-31 16:54:01,16.11,4,64.44,29.76,cash,Farleigh,Geach,Male,fgeach55@aol.com,296-345-9732,specialized,20 Fairfield Plaza,Portland,OR
1,McDavid Thigh Support,McDavid,Protective Gear,Thigh,NFL Properties LLC,2023-03-16 20:06:34,20.00,6,120.00,81.36,debit card,Stesha,Peiser,Female,speiserl@squarespace.com,562-102-4205,superstore,654 Pine View Place,New Orleans,LA


## Creación del Catálogo y el Schema

In [0]:
%sql
-- 1. Crear el Catálogo principal (si no existe)
CREATE CATALOG IF NOT EXISTS sportzone;

-- 2. Crear el Esquema de Ventas (base de datos)
CREATE SCHEMA IF NOT EXISTS sportzone.ventas_schema;

-- 3. Crear el Volume para almacenar archivos no tabulares,

CREATE VOLUME IF NOT EXISTS sportzone.ventas_schema.vol_ventas;

## Convertir df de pandas a Spark 

In [0]:
spark_df = spark.createDataFrame(df)

## Creación de la Tabla con Spark
1. Toma el conjunto de datos limpio contenido en el spark_df

2. Escribe esos datos en el formato Delta Lake dentro del almacenamiento en la nube (el Data Lake subyacente).

3. Registra esa nueva ubicación y estructura de datos bajo el nombre de la tabla **tbl_venta_spk**  en Unity Catalog (UC).

In [0]:
spark_df.write.mode("overwrite").saveAsTable("sportzone.ventas_schema.tbl_venta_spk")

## DESCRIBE TABLE 
En la tabla creada con spark

In [0]:
%sql
DESCRIBE TABLE sportzone.ventas_schema.tbl_venta_spk;

col_name,data_type,comment
product_name,string,null
brand,string,null
category,string,null
subcategory,string,null
supplier,string,null
date,string,null
price,double,null
quantity_sold,bigint,null
amount_sold,double,null
cost_amount,double,null


## Descripción de Datos Tabla creada con Spark 

In [0]:
# 1. Cargamos la tabla a un DataFrame
df_transacciones = spark.table("sportzone.ventas_schema.tbl_venta_spk")

# 2. Seleccionamos solo las columnas de tipo numérico (double, int, long)
#    y excluimos las de tipo string y timestamp.
numeric_cols = [
    f.name for f in df_transacciones.schema
    if f.dataType.typeName() in ('double', 'decimal', 'float', 'integer', 'long')
]

# 3. Creamos un nuevo DataFrame solo con esas columnas y aplicamos describe()
df_numeric_stats = df_transacciones.select(*numeric_cols)
display(df_numeric_stats.describe())

# df_numeric_stats.describe().show() # Usamos la función display que muestra de una mejor forma los resultados que .show().



summary,price,quantity_sold,amount_sold,cost_amount
count,501000,501000,501000,501000
mean,76.99570600798428,5.502864271457086,424.3352854890214,169.54772626746552
stddev,104.2215948087116,2.869251998464686,686.18905120711,273.3949835065696
min,1.99,1,2.05,0.8
max,800.96,10,8001.6,3203.1


## Validación de todas las tablas creadas dentro del sportzone.ventas_schema

In [0]:
%sql
SHOW TABLES IN sportzone.ventas_schema;



database,tableName,isTemporary
ventas_schema,tbl_venta,false
ventas_schema,tbl_venta_spk,false
,raw_csv_view,true


# 3. INGESTA DE KAGGLE
# CREACIÓN DE TABLA CON SQL

## 3.1. Obtención del dataset (Manualmente)
 Ruta Archivo:

/Volumes/sportzone/ventas_schema/vol_ventas/AF_offline_sales_dataset.csv

In [0]:
%sql
LIST '/Volumes/sportzone/ventas_schema/vol_ventas';


path,name,size,modification_time
/Volumes/sportzone/ventas_schema/vol_ventas/AF_offline_sales_dataset.csv,AF_offline_sales_dataset.csv,117358436,1763669720000


## 3.2. Carga en Spark
Lectura de la Data desde el Volumen con spark.read.csv sin crear la tabla


In [0]:
# 1. Lee la data directamente desde el Volume con Spark (sin crear la tabla)
ruta_csv_volume = 'dbfs:/Volumes/sportzone/ventas_schema/vol_ventas/AF_offline_sales_dataset.csv'

df_diagnostico = spark.read.csv(
  ruta_csv_volume,
  header=True,
  inferSchema=True
)

# 2. Imprime los nombres de las columnas que Spark REALMENTE leyó
print(df_diagnostico.columns)

['product_name', 'brand', 'category', 'subcategory', 'supplier', 'date', 'price', 'quantity_sold', 'amount_sold', 'cost_amount', 'payment_method', 'customer_firstname', 'customer_lastname', 'customer_gender', 'customer_email', 'customer_phone', 'store_type', 'store_street', 'store_city', 'store_state']


## 3.3. Persistencia

## Creación de una Vista Temporal

In [0]:
%sql
-- 1. Crear una vista temporal leyendo el archivo CSV desde el Volume
CREATE OR REPLACE TEMPORARY VIEW raw_csv_view 
USING CSV
OPTIONS (
  'path' = '/Volumes/sportzone/ventas_schema/vol_ventas/AF_offline_sales_dataset.csv',
  'header' = 'true',
  'inferSchema' = 'true',
  'timestampFormat' = 'yyyy-MM-dd HH:mm:ss'
);

-- 2. (Diagnóstico) Muestra las columnas reales leídas.
DESCRIBE raw_csv_view;

col_name,data_type,comment
product_name,string,null
brand,string,null
category,string,null
subcategory,string,null
supplier,string,null
date,timestamp,null
price,double,null
quantity_sold,int,null
amount_sold,double,null
cost_amount,double,null


## Creación de la Tabla con SQL

In [0]:
%sql
-- 2. Crear la tabla gestionada poblándola con los datos de la vista
CREATE TABLE IF NOT EXISTS sportzone.ventas_schema.tbl_venta
AS SELECT 
    product_name,
    brand,
    category,
    subcategory,
    supplier,
    date,
    price,
    quantity_sold,
    amount_sold,
    cost_amount,
    payment_method,
    customer_firstname,
    customer_lastname,
    customer_gender,
    customer_email,
    customer_phone,
    store_type,
    store_street,
    store_city,
    store_state
FROM raw_csv_view;

num_affected_rows,num_inserted_rows


## SELECT COUNT(*) 
Cantidad de Filas Cargadas 

In [0]:
%sql
SELECT COUNT(*) FROM sportzone.ventas_schema.tbl_venta;

COUNT(*)
501000


## DESCRIBE DETAIL
Muestra una tabla con metadatos como el format (Delta), el numFiles, y la location (la ruta interna donde UC guarda los archivos).


In [0]:
%sql
DESCRIBE DETAIL sportzone.ventas_schema.tbl_venta;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,dbc83317-8bb6-4e2c-9d05-3340c75df09f,sportzone.ventas_schema.tbl_venta,null,,2025-11-20T22:27:01.917Z,2025-11-20T22:27:07.000Z,List(),List(),2,21032925,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


# 4. VALIDACIONES: SPARK & SQL

## 4.1.  METADATOS

## a. Describe Table 
Muestra el esquema de la tabla Delta permanente tbl_venta

In [0]:
%sql
DESCRIBE TABLE sportzone.ventas_schema.tbl_venta;

col_name,data_type,comment
product_name,string,null
brand,string,null
category,string,null
subcategory,string,null
supplier,string,null
date,timestamp,null
price,double,null
quantity_sold,int,null
amount_sold,double,null
cost_amount,double,null


## b. Show  Create Table
Prueba que la tabla existe y está registrada permanentemente en el Catálogo, no solo en la memoria temporal.

In [0]:
%sql
SHOW CREATE TABLE sportzone.ventas_schema.tbl_venta;

createtab_stmt
"CREATE TABLE sportzone.ventas_schema.tbl_venta ( product_name STRING, brand STRING, category STRING, subcategory STRING, supplier STRING, date TIMESTAMP, price DOUBLE, quantity_sold INT, amount_sold DOUBLE, cost_amount DOUBLE, payment_method STRING, customer_firstname STRING, customer_lastname STRING, customer_gender STRING, customer_email STRING, customer_phone STRING, store_type STRING, store_street STRING, store_city STRING, store_state STRING) USING delta COLLATION 'UTF8_BINARY' TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


## c. Spark
spark_df.printSchema(), inspecciona y muestra la estructura del DataFrame de Spark.

In [0]:
spark_df.printSchema()

root
 |-- product_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- supplier: string (nullable = true)
 |-- date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity_sold: long (nullable = true)
 |-- amount_sold: double (nullable = true)
 |-- cost_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- customer_firstname: string (nullable = true)
 |-- customer_lastname: string (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_phone: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- store_street: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- store_state: string (nullable = true)



## 4.2.  DESCRIPCIÓN DE LOS DATOS  

## a. Descripción de Datos con df.describe().show()

In [0]:
# 1. Cargamos la tabla a un DataFrame
df_transacciones = spark.table("sportzone.ventas_schema.tbl_venta")

# 2. Seleccionamos solo las columnas de tipo numérico (double, int, long)
#    y excluimos las de tipo string y timestamp.
numeric_cols = [
    f.name for f in df_transacciones.schema
    if f.dataType.typeName() in ('double', 'decimal', 'float', 'integer', 'long')
]

# 3. Creamos un nuevo DataFrame solo con esas columnas y aplicamos describe()
df_numeric_stats = df_transacciones.select(*numeric_cols)
display(df_numeric_stats.describe())

# df_numeric_stats.describe().show() # Usamos la función display que muestra de una mejor forma los resultados que .show().



summary,price,quantity_sold,amount_sold,cost_amount
count,501000,501000,501000,501000
mean,76.99570600798518,5.502864271457086,424.3352854890265,169.54772626746552
stddev,104.22159480871163,2.869251998464686,686.1890512071102,273.3949835065695
min,1.99,1,2.05,0.8
max,800.96,10,8001.6,3203.1


## b. Descripción de Datos con SQL 

In [0]:
%sql
SELECT
    -- Fila 1: COUNT (Total de registros no nulos)
    'count' AS summary, 
    COUNT(price) AS price, 
    COUNT(quantity_sold) AS quantity_sold, 
    COUNT(amount_sold) AS amount_sold, 
    COUNT(cost_amount) AS cost_amount
FROM sportzone.ventas_schema.tbl_venta  -- Asegúrate que este es el nombre correcto

UNION ALL

SELECT 
    -- Fila 2: MEAN (Promedio)
    'mean' AS summary,
    ROUND(AVG(price), 4),
    ROUND(AVG(quantity_sold), 4),
    ROUND(AVG(amount_sold), 4),
    ROUND(AVG(cost_amount), 4)
FROM sportzone.ventas_schema.tbl_venta

UNION ALL

SELECT 
    -- Fila 3: STDDEV (Desviación Estándar)
    'stddev' AS summary,
    ROUND(STDDEV(price), 4),
    ROUND(STDDEV(quantity_sold), 4),
    ROUND(STDDEV(amount_sold), 4),
    ROUND(STDDEV(cost_amount), 4)
FROM sportzone.ventas_schema.tbl_venta

UNION ALL

SELECT 
    -- Fila 4: MIN (Valor Mínimo)
    'min' AS summary,
    MIN(price),
    MIN(quantity_sold),
    MIN(amount_sold),
    MIN(cost_amount)
FROM sportzone.ventas_schema.tbl_venta

UNION ALL

SELECT 
    -- Fila 5: MAX (Valor Máximo)
    'max' AS summary,
    MAX(price),
    MAX(quantity_sold),
    MAX(amount_sold),
    MAX(cost_amount)
FROM sportzone.ventas_schema.tbl_venta;

summary,price,quantity_sold,amount_sold,cost_amount
mean,76.9957,5.5029,424.3353,169.5477
stddev,104.2216,2.8693,686.1891,273.395
count,501000.0,501000.0,501000.0,501000.0
min,1.99,1.0,2.05,0.8
max,800.96,10.0,8001.6,3203.1


## 4.3. CONSULTAS EN SQL SELECT Y GROUP BY
Esta consulta muestra cuanto dinero se generó y el volumen de ventas por cada tipo de producto que se vende, tanto en sql como en Spark.

## a. Consulta con SQL

In [0]:
%sql
SELECT
    category,
    ROUND(SUM(amount_sold), 2) AS total_ingresos,
    ROUND(AVG(quantity_sold), 2) AS promedio_unidades_vendidas
FROM sportzone.ventas_schema.tbl_venta
GROUP BY category
ORDER BY total_ingresos DESC;

category,total_ingresos,promedio_unidades_vendidas
Helmets,7.252807064E7,5.51
Shoulder Pads,5.947285401E7,5.51
Gloves,2.626542383E7,5.5
Footwear,2.09727122E7,5.49
Clothing & Apparel,1.650503203E7,5.48
Protection,1.132197649E7,5.51
Protective Gear,3749219.35,5.5
Accessories,1776689.48,5.49


## b. Consulta con Spark 

In [0]:
from pyspark.sql.functions import sum, avg, round, col

# 1. Cargar la tabla Delta (la misma que usaste en SQL) en un DataFrame
df_ventas = spark.table("sportzone.ventas_schema.tbl_venta")

# 2. Agrupar y Agregar (GROUP BY y SUM/AVG)
reporte_por_categoria = df_ventas.groupBy("category").agg(
    # Calcula la suma total de amount_sold y la redondea
    round(sum(col("amount_sold")), 2).alias("total_ingresos"),
    # Calcula el promedio de quantity_sold y lo redondea
    round(avg(col("quantity_sold")), 2).alias("promedio_unidades_vendidas")
    
# 3. Ordenar por los ingresos totales de forma descendente (ORDER BY total_ingresos DESC)
).orderBy(col("total_ingresos").desc())

# 4. Mostrar el resultado como una tabla limpia
display(reporte_por_categoria)

category,total_ingresos,promedio_unidades_vendidas
Helmets,7.252807064E7,5.51
Shoulder Pads,5.947285401E7,5.51
Gloves,2.626542383E7,5.5
Footwear,2.09727122E7,5.49
Clothing & Apparel,1.650503203E7,5.48
Protection,1.132197649E7,5.51
Protective Gear,3749219.35,5.5
Accessories,1776689.48,5.49


## 4.4. CONTEOS Y MUESTRAS 

## a. Conteo de Registros (COUNT(*))
Cuántos registros totales hay, cuántos clientes diferentes hay, y cuántos productos diferentes se vende.



In [0]:
%sql
SELECT 
    COUNT(*) AS total_filas_cargadas,
    COUNT(DISTINCT customer_email) AS clientes_unicos,
    COUNT(DISTINCT product_name) AS productos_unicos
FROM sportzone.ventas_schema.tbl_venta;

total_filas_cargadas,clientes_unicos,productos_unicos
501000,365000,573


## b. Muestra y Limitación de Datos (SELECT * LIMIT)
Se usa el comando LIMIT para obtener una muestra rápida del conjunto de datos.

In [0]:
%sql
SELECT
    product_name,
    brand,
    category,
    date,
    price,
    quantity_sold
FROM sportzone.ventas_schema.tbl_venta
LIMIT 10;

product_name,brand,category,date,price,quantity_sold
Schutt XV HD QB/WR Shoulder pad,Schutt,Shoulder Pads,2023-11-22T08:48:52.000Z,298.09,8
Shock Doctor Max AirFlow LG,Shock Doctor,Protection,2022-12-16T07:04:12.000Z,6.74,5
Shock Doctor Shield,Shock Doctor,Protection,2022-11-23T05:03:33.000Z,5.21,8
Gridirons wristband,Markwort,Accessories,2022-08-30T04:10:07.000Z,9.23,4
Shock Doctor 2 Pack Shields,Shock Doctor,Protection,2022-04-20T17:39:49.000Z,4.75,10
Cutters SO17JE JE11 Signature Series,Cutters,Gloves,2023-09-14T08:43:19.000Z,85.96,1
Riddell Speed Icon Threaded Valve Retainer Cap (R929901),Riddell,Helmets,2023-10-24T01:43:55.000Z,6.33,10
Schutt Mid Flex 4.0 All Purpose Youth Shoulder Pads,Schutt,Shoulder Pads,2022-12-05T17:32:45.000Z,133.76,7
3DX Jaw Guard Xenith,Xenith,Helmets,2023-12-16T09:19:09.000Z,49.88,10
Rawlings SO2RH Hollow Wire,Rawlings,Helmets,2023-06-12T17:50:37.000Z,55.21,9


# 5. SQL VS SPARK: VENTAJAS Y DESVENTAJAS

![Ventajas y Desventajas SQL vs Spark](./VD.png)